# Calculate radiative heating profile

## Import packages

In [1]:
import os
import sys
import h5py
import numpy as np

from dataclasses import asdict
from typing import cast
from pathlib import Path

sys.path.append("/home/b11209013/Kuang-python/src")
from Configuration.Params import Radiation, Constant
from Configuration.Domain import Domain
from ModelCalculation.MeanState import MeanState

## Load data

In [2]:
# Path setup
data_path : Path = Path("/work/b11209013/Kuang-python/v0.1.0/two-way/Eastward/Rad/2026-10-28-08-37-05-996147668-3800833")

# Output path
output_path: Path = Path("/home/b11209013/Kuang-python/v0.1.0_Analysis/output/v0.1.0/Radiation")
output_file: Path = output_path / f"{data_path.parent.name}_{data_path.name}_Radiation.h5"
os.makedirs(output_path, exist_ok=True)

# Load vertical motions
with h5py.File(data_path / "FourierState.h5", "r") as f:
    x   : np.ndarray = cast(h5py.Dataset, f["x"])[:]
    time: np.ndarray = cast(h5py.Dataset, f["time"])[:]

    w1  : np.ndarray = cast(h5py.Dataset, f["w1"])[:].real # shape: (n_ensemble, nt, nx)
    w2  : np.ndarray = cast(h5py.Dataset, f["w2"])[:].real

    nens, nt, nx = w1.shape

# Extract variables through class objects
rad_params : dict[str, float] = asdict(Radiation())
const      : Constant = Constant()
domain     : Domain = Domain()
mean_state : MeanState = MeanState()
rad_scaling: float = 1.0 # Radiation strength of the selected experiment

## Calculate Fourier coefficient of radiative heating.

In [3]:
# LW Fourier coefficient
LW1: np.ndarray = rad_scaling * (rad_params["alpha11_LW"] * w1 + rad_params["alpha12_LW"] * w2)
LW2: np.ndarray = rad_scaling * (rad_params["alpha21_LW"] * w1 + rad_params["alpha22_LW"] * w2)

# SW Fourier coefficient
SW1: np.ndarray = rad_scaling * (rad_params["alpha11_SW"] * w1 + rad_params["alpha12_SW"] * w2)
SW2: np.ndarray = rad_scaling * (rad_params["alpha21_SW"] * w1 + rad_params["alpha22_SW"] * w2)

## Reconstruct vertical profile

Use the density and stratification factors from the model's heating reconstruction.
The profile axes are `(ensemble, time, x, z)` for the following composite analysis.

In [4]:
# Galerkin basis
z_coords: np.ndarray = domain.Z
G1: np.ndarray = np.pi/2 * np.sin(np.pi*z_coords / z_coords.max())
G2: np.ndarray = np.pi/2 * np.sin(2 * np.pi*z_coords / z_coords.max())

# Convert modal heating to K/day, as for J1 and J2 in the model
rho          : np.ndarray = mean_state.rhomean()
stratification: float = const.g / const.cp - const.GammaT

# Construct vertical profile; shape: (ensemble, time, x, z)
LW1_vert: np.ndarray = np.einsum("etx,z->etxz", LW1, G1 * stratification / rho)
LW2_vert: np.ndarray = np.einsum("etx,z->etxz", LW2, G2 * stratification / rho)
SW1_vert: np.ndarray = np.einsum("etx,z->etxz", SW1, G1 * stratification / rho)
SW2_vert: np.ndarray = np.einsum("etx,z->etxz", SW2, G2 * stratification / rho)

print("Shape of LW1_vert:", LW1_vert.shape)

Shape of LW1_vert: (90, 40, 401, 71)


## Save radiative heating profile

Save `LW1`, `LW2`, `SW1`, `SW2` in K/day with `x`, `z`, and `time`.
Total LW is `LW1 + LW2`; total SW is `SW1 + SW2`.

In [5]:
# Save profiles and coordinates for the following composite analysis
with h5py.File(output_file, "w") as f:
    f.attrs["complete"] = False
    f.attrs["source_directory"] = str(data_path)
    f.attrs["axis_order"] = "ensemble, time, x, z"
    f.attrs["rad_scaling"] = rad_scaling

    for name, values, units in [("x", x, "m"), ("z", z_coords, "m"), ("time", time, "day")]:
        dataset = f.create_dataset(name, data=values)
        dataset.attrs["units"] = units

    for name, values in [("LW1", LW1_vert), ("LW2", LW2_vert), ("SW1", SW1_vert), ("SW2", SW2_vert)]:
        dataset = f.create_dataset(name, data=values, compression="gzip", compression_opts=1, shuffle=True)
        dataset.attrs["units"] = "K/day"
        dataset.attrs["axis_order"] = "ensemble, time, x, z"

    f.attrs["complete"] = True

print(f"Saved: {output_file}")

Saved: /home/b11209013/Kuang-python/v0.1.0_Analysis/output/v0.1.0/Radiation/Rad_2026-10-28-08-37-05-996147668-3800833_Radiation.h5
